# Combining Data Wrap-up
v.ekc

Today we cover specialized pandas tools for joining time series at different frequencies and approximating temporal matches. A lot of the time, times will not perfectly line up! We will also look at geographic lat/long data

| Section | Topic |
|---------|-------|
| 1 | Setup |
| 2 | Combining Time Series Data |
| 3 | Approximate Temporal Matches with merge_asof |
| 4 | Spatial + Temporal Matching |
| 5 | 🔬 Activity: Bird GPS Data |
| Appendix | Quick Reference |

---
## 1. Setup

In [ ]:
import numpy as np
import pandas as pd

---
## 2. Combining Time Series Data

When time series data comes from different sensors or at different frequencies, a plain `.merge()` may silently sort incorrectly or leave gaps. `merge_ordered` and `resample` give you more control.



| Function | What it does |
|----------|--------------|
| `pd.merge_ordered(df1, df2)` | Merge and sort by the key column |
| `pd.merge_ordered(..., fill_method='ffill')` | Forward-fill gaps after merging |
| `df.resample('1h', on='time').sum()` | Aggregate to a coarser time resolution |
| `df.groupby(pd.Grouper(key='time', freq='1h')).sum()` | Group by time — safe when other cols shouldn't be summed |

Below we will load two environmental sensor datasets that **don't share the same timestamps**. One sensor logs hourly and another logs on a different schedule.

> **Why does this matter?** A plain `.merge()` only keeps rows where both timestamps match exactly. If your sensors are even slightly out of sync, you silently lose data.

In [ ]:
env_data = pd.DataFrame({
    'time': pd.date_range('2025-04-14 08:00', periods=6, freq='1h'),
    'temperature_C': [14.0, 12.5, 13.0, 13.5, 12.0, 14.2],
    'humidity_pct': [85, 83, 80, 78, 76, 75]
})
env_data.sort_values(by='temperature_C',inplace=True)
env_data

In [ ]:
precip_data = pd.DataFrame({
    'time': pd.date_range('2025-04-14 07:00', periods=9, freq='1h'),
    'precipitation': [0.1, 0.3, 0, 0, 0.1, 0.05, 0.05, 0.1, 0.1],
})
precip_data

In [ ]:
# A standard merge goes in the order of the original left dataframe
pd.merge(env_data, precip_data)

In [ ]:
# merge_ordered sorts the data too
pd.merge_ordered(env_data, precip_data)

In [ ]:
# merge_ordered also allows you to fill in gaps with a method of choice
pd.merge_ordered(env_data, precip_data, fill_method='ffill')

The examples above worked because both DataFrames happened to use the **same frequency** (hourly). The real power of `merge_ordered` shows up when the two datasets have **different time steps**. For example, hourly environment readings and 30-minute precipitation readings.

In [ ]:
# When this is most useful?
precip_data = pd.DataFrame({
    'time': pd.date_range('2025-04-14 07:00', periods=10, freq='30min'),
    'precipitation': [0.1, 0.3, 0, 0, 0.1, 0.05, 0.05, 0.1, 0.1, 0.2],
})
precip_data

In [ ]:
# When the frequency of data points is different
env_data

In [ ]:
# When the frequency of data points is different
pd.merge_ordered(env_data, precip_data)

In [ ]:
pd.merge_ordered(env_data, precip_data, fill_method='ffill')

`merge_ordered` also accepts the same `how` argument as a regular merge. An `inner` join keeps only timestamps where **both** DataFrames have an observation — useful when you only care about moments of overlap.

In [ ]:
# Merge_ordered also accepts different "how" args
pd.merge_ordered(env_data, precip_data, how = 'inner')

### An alternative approach: coarsen first, then merge

Instead of merging at the finer resolution and forward-filling, you can **aggregate the high-frequency data down to the lower frequency** with `resample`, then do a clean merge. This is simpler when you genuinely don't need sub-hourly detail.

In [ ]:
# You can also take the approach of "course graining" with resample
hourly_precip = precip_data.resample('1h', on='time').sum()
hourly_precip

In [ ]:
# Merge at course-grained resolution
course = pd.merge_ordered(env_data,hourly_precip.reset_index())
course

#### Watch out: `resample` aggregates *every* numeric column

The approach above is fine for a single numeric column. But once the DataFrame has **location columns** (`lat`, `lon`), `resample().sum()` will blindly sum those too — which is physically meaningless. Let's see that failure, then fix it.

In [ ]:
# When should we be careful about resampling?
precip_data['lat'] = [38.1, 38.1, 38.1, 38.1, 38.1, 38.2, 38.2, 38.2, 38.2, 38.2]
precip_data['lon'] = [-120.1, -120.1, -120.2,-120.2, -120.2,-120.1, -120.1, -120.2,-120.2, -120.2]
precip_data

In [ ]:
# Resampling this would lead to inaccurate location data
precip_data.resample('1h',on='time').sum()

Summing latitude and longitude is nonsense. The fix is `pd.Grouper` inside a `groupby` — it gives you time-based grouping **without touching columns you didn't explicitly aggregate**.

In [ ]:
# In this case, use groupby with pd.Grouper
precip_data.groupby(pd.Grouper(key = 'time', freq="1h")).sum()

In [ ]:
# Groupby can handle multiple columns
precip_data.groupby([pd.Grouper(key = 'time', freq="1h"),'lat','lon']).sum()

In [ ]:
# Get course-grained data for each location
precip_data.groupby([pd.Grouper(key = 'time', freq="1h"),'lat','lon']).mean().reset_index()

---
### Try It 1: Merging and Resampling Time Series 😊

Use the `env_data` and `precip_data` DataFrames from above (the version with `lat`/`lon` columns).

1. Use `merge_ordered` to merge `env_data` and `precip_data`, then forward-fill any gaps with `fill_method='ffill'`. How many rows does the result have?

2. Instead of forward-filling, try `how='inner'` — how does this change the number of rows?

3. `precip_data` has `lat` and `lon` columns. Try `precip_data.resample('1H', on='time').sum()` — what goes wrong with the location columns? Fix it by using `pd.Grouper`.

In [ ]:
# 1. merge_ordered with ffill


In [ ]:
# 2. merge_ordered with how='inner'


In [ ]:
# 3. resample (observe the problem), then fix with pd.Grouper


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*`resample().sum()` sums the lat/lon columns too, which is nonsensical. Use `pd.Grouper` to aggregate only what makes sense.*

```python
# 1.
pd.merge_ordered(env_data, precip_data, fill_method='ffill')

# 2.
pd.merge_ordered(env_data, precip_data, how='inner')
# Fewer rows — only timestamps present in BOTH datasets

# 3. Problem:
precip_data.resample('1H', on='time').sum()  # lat/lon get summed — meaningless!

# Fix:
precip_data.groupby([pd.Grouper(key='time', freq='1H'), 'lat', 'lon'])['precipitation'].sum().reset_index()
```

</details>

---
## 3. Approximate Temporal Matches — merge_asof

What if your two datasets have timestamps that never align exactly? `merge_asof` matches each left timestamp to the nearest (or most recent) right timestamp.



| What | Code |
|------|------|
| Match to most recent prior event | `pd.merge_asof(left, right, on='time')` |
| Match to nearest event | `pd.merge_asof(..., direction='nearest')` |
| Match to next future event | `pd.merge_asof(..., direction='forward')` |
| Different column names | `pd.merge_asof(..., left_on='t1', right_on='t2')` |
| Require close match | `pd.merge_asof(..., tolerance=pd.Timedelta('1h'))` |

Let's build two datasets that can never be joined on exact timestamps: wildlife **observations** logged the moment a researcher spots an animal, and **environmental sensor readings** taken on a fixed hourly schedule. These two will almost never share an exact time.

In [ ]:
# Irregular observational data
observations = pd.DataFrame({
    'time': pd.to_datetime([
        '2025-04-14 08:03', '2025-04-14 09:21', '2025-04-14 11:00',
        '2025-04-14 09:45', '2025-04-14 13:27'
    ]),
    'animal': ['fox', 'bear', 'wolf', 'bear', 'wolf'],
    'location': ['ridge', 'valley', 'creek', 'creek', 'ridge']
}).sort_values('time')

observations

In [ ]:
env_data = pd.DataFrame({
    'time': pd.date_range('2025-04-14 08:00', periods=6, freq='1h'),
    'temperature_C': [12.0, 12.5, 13.0, 13.5, 14.0, 14.2],
    'humidity_pct': [85, 83, 80, 78, 76, 75]
})
env_data

In [ ]:
# Merge ordered takes all unique
pd.merge_ordered(observations, env_data)

In [ ]:
# Merge ordered incorrectly duplicates observations
pd.merge_ordered(observations, env_data, fill_method='ffill')

Both approaches above are broken for this use case:
- `merge_ordered` **without** fill creates rows for every unique timestamp — so each observation gets its own row with `NaN` environment data.
- `merge_ordered` **with** `ffill` duplicates the observation rows instead of filling the gaps.

What we actually want: *for each observation, find the closest env reading without duplicating anything.* That's exactly what **`merge_asof`** does.

In [ ]:
# Remind ourselves of what the originals look like
observations

In [ ]:
env_data

In [ ]:
# Merge the left time points to the previous point from the right
pd.merge_asof(observations, env_data, on='time')

In [ ]:
# Merge the left time points to the nearest point from the right 
pd.merge_asof(observations, env_data, on='time', direction='nearest')

In [ ]:
# Merge the left time points to the next point from the right
pd.merge_asof(observations, env_data, on='time', direction='forward')

#### What if the time columns have different names?

So far both DataFrames used a column called `'time'`. In practice, one dataset might call it `'timestamp'` or `'Time'`. Use `left_on` / `right_on` to handle that — just like in a regular merge.

In [ ]:
# If they have different names
env_data.rename(columns={'time':'Time'},inplace=True)
pd.merge_asof(observations, env_data, left_on='time', right_on = 'Time')

In [ ]:
# Reset
env_data.rename(columns={'Time':'time'},inplace=True)

---
### Try It 2: Choosing a merge_asof Direction 😊

Use the `observations` and `env_data` DataFrames (before lat/lon are added).

1. Run `merge_asof(observations, env_data, on='time')` with the default direction. Which temperature reading does the **09:21 bear** observation get, and why?

2. Switch to `direction='forward'`. Which observation now has a `NaN` temperature, and why?

3. Use `direction='nearest'` with `tolerance=pd.Timedelta('30min')`. How many `NaN` temperature values appear? Which observations lost their match?

In [ ]:
# 1. Default (backward) direction


In [ ]:
# 2. direction='forward'


In [ ]:
# 3. direction='nearest' with 30-min tolerance


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Backward (default) attaches the most recent prior reading — sensible when you want the conditions leading up to an observation.*
*Forward attaches the next future reading — the 13:27 wolf observation gets NaN because no env reading exists at 14:00 or later.*

```python
# 1.
pd.merge_asof(observations, env_data, on='time')
# 09:21 bear → temperature at 09:00 (most recent prior reading)

# 2.
pd.merge_asof(observations, env_data, on='time', direction='forward')
# 13:27 wolf → NaN (no env reading at 14:00+)

# 3.
pd.merge_asof(observations, env_data, on='time',
              direction='nearest', tolerance=pd.Timedelta('30min'))
# 11:00 wolf → NaN (nearest env point is 11:00, within 30 min — wait, that's fine!)
# 13:27 wolf → NaN (nearest env point is 13:00, 27 min away — still within tolerance)
# 09:21 bear → NaN (nearest is 09:00, 21 min away — within tolerance)
# Result: only obs more than 30 min from any reading get NaN
```

</details>

---
## 4. Spatial + Temporal Matching

When data has both location and time, you can constrain `merge_asof` to match within the same spatial group — and even set a time tolerance.

In [ ]:
observations['lat'] = [38.2, 38.2, 38.2, 38.2, 38.2]
observations['lon'] = [-120.1, -120.1, -120.2,-120.2, -120.2]

In [ ]:
env_data['lat'] = [38.2, 38.2, 38.2, 38.2, 38.2,38.2]
env_data['lon'] = [-120.1, -120.1, -120.2,-120.2, -120.2, -120.1]

Now both DataFrames have location information. This raises a new problem: an observation at location A shouldn't be matched to an env reading at location B, even if that reading is the closest in time. We need a **spatial constraint**.

In [ ]:
observations

In [ ]:
env_data

In [ ]:
# Merge each location on the time column
pd.merge_asof(observations, env_data, on='time')

Without `by`, every observation gets matched to the nearest env reading regardless of location — a fox spotted at one coordinates gets temperature data from a sensor kilometers away. The `by` parameter fixes this by requiring that matches share the **same values** in those columns.

In [ ]:
pd.merge_asof(observations, env_data, on='time', by = ['lat','lon'])

When you add `by`, some observations may no longer find *any* match.....if no env reading at that exact location exists before the observation time, the result is `NaN`. You can make the constraint **slightly more lenient** with a time `tolerance`: only match if the closest reading is within a given window.

In [ ]:
# Set a tolerance for how close the points have to be to match
pd.merge_asof(observations, env_data, on='time', by = ['lat','lon'],
              tolerance=pd.Timedelta('1h'))

---
### Try It 3: Spatial Constraints with merge_asof 😊

Use the `observations` and `env_data` DataFrames after the lat/lon columns have been added.

1. Run `merge_asof(observations, env_data, on='time')` **without** `by`. Do all rows get a temperature match?

2. Add `by=['lat', 'lon']` — which rows now show `NaN` for temperature? Why does restricting to location cause some matches to disappear?

3. Add `tolerance=pd.Timedelta('2h')` together with `by=['lat', 'lon']`. Does the larger time window bring back any matches that were `NaN` in step 2?

In [ ]:
# 1. merge_asof without spatial constraint


In [ ]:
# 2. Add by=['lat', 'lon']


In [ ]:
# 3. Add tolerance=pd.Timedelta('2h') with by=['lat', 'lon']


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Without `by`, every observation matches some env reading. Adding `by` enforces that only env readings at the **same location** can be matched — if no prior env reading at that lat/lon exists, the match is NaN.*

```python
# 1.
pd.merge_asof(observations, env_data, on='time')  # all rows matched

# 2.
pd.merge_asof(observations, env_data, on='time', by=['lat', 'lon'])
# Rows where no prior env reading shares the same lat/lon → NaN

# 3.
pd.merge_asof(observations, env_data, on='time',
              by=['lat', 'lon'], tolerance=pd.Timedelta('2h'))
# A larger tolerance only helps if a match at the same location exists within 2h
# It does NOT recover matches where the lat/lon combination is absent from env_data
```

</details>

---
## 5. 🔬 Activity — Bird GPS Data

You're analyzing bird behavior with GPS trackers and sound-activated recorders. The GPS trackers log positions every 5 minutes. The sound recorders log audio events (calls, songs, other birds) at irregular times.

For each bird location, merge the most recent sound event recorded in that area.

---
### Try It 4: Merge Bird GPS + Sound Events 😊

1. **Inspect** `bird_gps` and `sound_events` — what are the timestamps and how do they align?

2. **Use `merge_asof`** to attach the most recent sound event to each GPS position.

3. **Try `direction='nearest'`** — does the result change? Which direction makes more biological sense for this use case?

In [ ]:
bird_gps = pd.DataFrame({
    "time": pd.to_datetime([
        "2023-04-13 08:00", "2023-04-13 08:05", "2023-04-13 08:10"
    ]),
    "bird_id": [101, 101, 101],
    "lat": [40.5, 40.5, 40.5],
    "lon": [-123.8, -123.8, -123.8]
})

sound_events = pd.DataFrame({
    "time": pd.to_datetime([
        "2023-04-13 07:59", "2023-04-13 08:03", "2023-04-13 08:12"
    ]),
    "event_type": ["call", "song", "flight"]
})

In [ ]:
# 1. Inspect the two datasets


In [ ]:
# 2. merge_asof: most recent sound event per GPS point


In [ ]:
# 3. Try direction='nearest'


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*`direction='backward'` (default) makes more biological sense — we attach the sound that was recorded before the bird's position was logged, not after.*

```python
# 2. Most recent prior sound event
pd.merge_asof(bird_gps, sound_events, on='time')

# 3. Nearest sound event
pd.merge_asof(bird_gps, sound_events, on='time', direction='nearest')
# The 08:10 GPS point now gets 'flight' (at 08:12) instead of 'song' (at 08:03)
```

</details>

---
## Appendix — Time Series Combining Quick Reference

### merge_ordered
```python
pd.merge_ordered(df1, df2)                          # merge and sort by key
pd.merge_ordered(df1, df2, fill_method='ffill')     # forward-fill gaps
pd.merge_ordered(df1, df2, how='inner')             # inner join, sorted
```

### Resampling
```python
df.resample('1H', on='time').sum()                  # coarsen to hourly totals
# Use groupby when other columns shouldn't be aggregated:
df.groupby(pd.Grouper(key='time', freq='1H')).sum()
df.groupby([pd.Grouper(key='time', freq='1H'), 'lat', 'lon']).mean()
```

### merge_asof
```python
pd.merge_asof(left, right, on='time')               # match to most recent prior
pd.merge_asof(left, right, on='time',
              direction='nearest')                  # match to nearest
pd.merge_asof(left, right, on='time',
              direction='forward')                  # match to next future
pd.merge_asof(left, right, on='time',
              by=['lat','lon'])                      # match within groups
pd.merge_asof(left, right, on='time',
              tolerance=pd.Timedelta('1h'))          # require close match
```